# Lab 4: Classification and Regression

## 1. Problem Framing
- **Classification:** Predict whether a support ticket will be escalated (`escalated`).
- **Regression:** Predict the time required to resolve a support ticket (`resolution_time_hours`).
The objective is to compare different machine learning models, evaluate their performance, and select the most suitable models.

In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import (
    LogisticRegression,
    LinearRegression,
    Ridge
)

from sklearn.neighbors import KNeighborsClassifier

from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error
)

print("All libraries imported successfully!")


All libraries imported successfully!


In [2]:

# Load the classification dataset
clf_df = pd.read_csv("data/classification_escalation.csv")

# Load the regression dataset
reg_df = pd.read_csv("data/resolution_time_regression.csv")

# Display the first five rows of each dataset
print("CLASSIFICATION DATASET")
display(clf_df.head())

print("REGRESSION DATASET")
display(reg_df.head())


CLASSIFICATION DATASET


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


REGRESSION DATASET


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [3]:

print("Dataset shape:", clf_df.shape)

print("\nColumn names:")
print(clf_df.columns.tolist())

print("\nDataset information:")
clf_df.info()

print("\nMissing values:")
print(clf_df.isnull().sum())

print("\nEscalation distribution:")
print(clf_df["escalated"].value_counts())


Dataset shape: (900, 7)

Column names:
['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'escalated']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB

Missing values:
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    3

In [4]:

# Separate features (X) and target (y)
X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


Training features: (720, 5)
Testing features: (180, 5)

Training target distribution:
escalated
0    567
1    153
Name: count, dtype: int64

Testing target distribution:
escalated
0    142
1     38
Name: count, dtype: int64


## 2. Experiment A: Classification

The `ticket_id` column was removed because it is only an identifier. Missing numerical values were replaced with the median, and missing categorical values with the most frequent value. Numerical features were scaled using StandardScaler, while categorical features were encoded using OneHotEncoder.

The data was split into 80% training and 20% testing data using stratification to preserve the class proportions. Five-fold Stratified Cross-Validation was used to compare Logistic Regression, KNN, and Decision Tree, using F1-score as the primary metric because the target classes are imbalanced.

In [5]:

# Identify numerical and categorical columns
numeric_features = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months"
]

categorical_features = [
    "priority",
    "channel"
]

# Preprocessing for numerical data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [6]:

# Define the three classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# Create a pipeline for each model
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

print("All three classification pipelines created successfully!")


All three classification pipelines created successfully!


In [7]:

# Set up 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold Stratified Cross-Validation is ready!")


5-fold Stratified Cross-Validation is ready!


In [8]:

# Compare classification models using 5-fold cross-validation

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

classification_results = []

for name, pipeline in pipelines.items():

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    classification_results.append({
        "Model": name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std()
    })

# Display results
classification_results_df = pd.DataFrame(classification_results)

display(classification_results_df.round(3))


,Model,Accuracy Mean,Precision Mean,Recall Mean,F1 Mean,F1 Std
0,Logistic Regression,0.812,0.624,0.288,0.393,0.063
1,KNN,0.800,0.562,0.268,0.361,0.050
2,Decision Tree,0.704,0.318,0.347,0.331,0.058


In [9]:

# Define the hyperparameters to test
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=pipelines["Logistic Regression"],
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

# Train and search for the best settings
grid_search.fit(X_train, y_train)

# Display the best results
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1:", round(grid_search.best_score_, 3))


Best Parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Best Cross-Validation F1: 0.52


In [10]:

# Get the best tuned model
best_clf = grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_clf.predict(X_test)

# Display classification report
print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))

# Display confusion matrix
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180


CONFUSION MATRIX
[[117  25]
 [ 11  27]]


### Classification Results

Logistic Regression achieved the highest initial CV F1-score (0.393), so it was selected for tuning. GridSearchCV tested different `C` and class-weight values. The best settings were `C = 0.1` and `class_weight = balanced`, with a CV F1-score of 0.520.

The final test accuracy was 0.80, with a recall of 0.71 for escalated tickets. The model identified 27 escalated tickets, missed 11, and produced 25 false positives. It is useful for identifying escalations but still makes some incorrect predictions.

In [11]:

# Check dataset dimensions
print("Dataset shape:", reg_df.shape)

# Check columns and data types
print("\nDataset information:")
reg_df.info()

# Check missing values
print("\nMissing values:")
print(reg_df.isnull().sum())

# Examine target variable
print("\nResolution time statistics:")
print(reg_df["resolution_time_hours"].describe())


Dataset shape: (800, 7)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               800 non-null    str    
 1   complexity_score        784 non-null    float64
 2   team_experience_years   776 non-null    float64
 3   priority                800 non-null    str    
 4   channel                 800 non-null    str    
 5   customer_tenure_months  780 non-null    float64
 6   resolution_time_hours   800 non-null    float64
dtypes: float64(4), str(3)
memory usage: 43.9 KB

Missing values:
ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64

Resolution time statistics:
count    800.000000
mean       5.873160
std        4.144974
min        0.

In [12]:

# Separate features and target
X_reg = reg_df.drop(
    columns=["ticket_id", "resolution_time_hours"]
)

y_reg = reg_df["resolution_time_hours"]

# Split into training and testing sets
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_reg_train.shape)
print("Testing features:", X_reg_test.shape)

print("\nTraining target:", y_reg_train.shape)
print("Testing target:", y_reg_test.shape)


Training features: (640, 5)
Testing features: (160, 5)

Training target: (640,)
Testing target: (160,)


## 3. Experiment B: Regression

The `ticket_id` column was removed because it is only an identifier. Missing numerical values were replaced with the median and categorical values with the most frequent value. StandardScaler and OneHotEncoder were used for numerical and categorical features respectively.

The data was split into 80% training and 20% testing data. Five-fold K-Fold Cross-Validation was used to compare Linear Regression, Ridge Regression, and Decision Tree Regression using MAE.

In [13]:

# Numerical preprocessing
numeric_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor_reg = ColumnTransformer(transformers=[
    ("num", numeric_transformer_reg, numeric_features),
    ("cat", categorical_transformer_reg, categorical_features)
])

print("Regression preprocessing pipeline created successfully!")


Regression preprocessing pipeline created successfully!


In [14]:

# Define regression models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

# Create a pipeline for each model
reg_pipelines = {}

for name, model in reg_models.items():
    reg_pipelines[name] = Pipeline(steps=[
        ("preprocessor", preprocessor_reg),
        ("regressor", model)
    ])

print("All three regression pipelines created successfully!")


All three regression pipelines created successfully!


In [15]:

# Set up 5-fold cross-validation
reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold regression cross-validation is ready!")


5-fold regression cross-validation is ready!


In [16]:

# Compare regression models using MAE
regression_results = []

for name, pipeline in reg_pipelines.items():

    scores = cross_validate(
        pipeline,
        X_reg_train,
        y_reg_train,
        cv=reg_cv,
        scoring="neg_mean_absolute_error"
    )

    mae_scores = -scores["test_score"]

    regression_results.append({
        "Model": name,
        "MAE Mean": mae_scores.mean(),
        "MAE Std": mae_scores.std()
    })

# Display results
regression_results_df = pd.DataFrame(regression_results)

display(regression_results_df.round(3))


,Model,MAE Mean,MAE Std
0,Linear Regression,2.035,0.172
1,Ridge Regression,2.035,0.171
2,Decision Tree,3.239,0.054


In [17]:

# Define Ridge hyperparameter values
ridge_param_grid = {
    "regressor__alpha": [0.01, 0.1, 1, 10, 100]
}

# Set up GridSearchCV
ridge_grid = GridSearchCV(
    estimator=reg_pipelines["Ridge Regression"],
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=reg_cv,
    n_jobs=-1
)

# Train and find the best alpha
ridge_grid.fit(X_reg_train, y_reg_train)

# Display the best results
print("Best Alpha:", ridge_grid.best_params_)
print("Best Cross-Validation MAE:", round(-ridge_grid.best_score_, 3))


Best Alpha: {'regressor__alpha': 0.1}
Best Cross-Validation MAE: 2.035


In [18]:

from sklearn.metrics import mean_squared_error

# Get the best Ridge model
best_reg = ridge_grid.best_estimator_

# Make predictions on the test data
y_reg_pred = best_reg.predict(X_reg_test)

# Calculate test errors
test_mae = mean_absolute_error(y_reg_test, y_reg_pred)
test_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("FINAL REGRESSION TEST RESULTS")
print("Test MAE:", round(test_mae, 3), "hours")
print("Test RMSE:", round(test_rmse, 3), "hours")


FINAL REGRESSION TEST RESULTS
Test MAE: 2.23 hours
Test RMSE: 3.403 hours


In [19]:

# Check for outliers using the IQR method
Q1 = y_reg_train.quantile(0.25)
Q3 = y_reg_train.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = y_reg_train[
    (y_reg_train < lower_bound) |
    (y_reg_train > upper_bound)
]

print("First Quartile (Q1):", round(Q1, 2))
print("Third Quartile (Q3):", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))
print("Number of possible outliers:", len(outliers))


First Quartile (Q1): 2.57
Third Quartile (Q3): 8.16
IQR: 5.58
Lower Bound: -5.8
Upper Bound: 16.53
Number of possible outliers: 8


### Regression Results

Linear Regression and Ridge Regression achieved the lowest CV MAE of 2.035 hours. Ridge was selected because it had slightly lower fold variability. GridSearchCV tested alpha values of 0.01, 0.1, 1, 10, and 100. The best alpha was 0.1.

The final test MAE was 2.23 hours and RMSE was 3.403 hours. The higher RMSE indicates that some predictions may have larger errors. The IQR method identified eight possible outliers in the training data, which may contribute to these errors.

Ridge Regression was selected for its low prediction error, although it may struggle with unusual resolution times.

**Leakage Prevention**

Preprocessing was placed inside Scikit-learn pipelines to prevent data leakage. This ensures that imputers, scalers, and encoders are fitted only on training data within each cross-validation fold. The test data was kept separate and used only for final evaluation.
1. Why stratify the classification split?

To maintain approximately the same proportion of each target class in the training and testing datasets, especially because the classification dataset is imbalanced.

2. Why must preprocessing be inside the pipeline?

To prevent data leakage by ensuring preprocessing learns only from training data within each cross-validation fold.

3. What does fold variability tell you?

It shows how consistent a model's performance is across different folds. Lower standard deviation indicates more consistent performance.

4. Why should the test set be evaluated only once?

To keep the test data independent and provide an unbiased final evaluation. Repeated testing and model adjustments could lead to overfitting.

5. What deployment constraints might change your choice?

Prediction speed, computational resources, explainability, the cost of false positives and false negatives, and performance on new data could all influence model selection.

